In [4]:
import networkx as nx
import pandas as pd

In [90]:
# Load dataset
DIR = "/REF_GRA/DATA/enerpol_full.pickle"
df = pd.read_pickle(DIR)

In [100]:
# Dataset info
# Filter no references

df = df[df["References"] != "no_references"]
df = df[df["Authors"] != "no_author"]

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 828 entries, 0 to 10
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Title                     828 non-null    object
 1   Authors_and_Affiliations  828 non-null    object
 2   Affiliations              828 non-null    object
 3   DOI                       828 non-null    object
 4   Authors                   828 non-null    object
 5   Journal                   828 non-null    object
 6   Date                      828 non-null    object
 7   Subjects                  828 non-null    object
 8   Abstract                  828 non-null    object
 9   References                828 non-null    object
 10  Content                   828 non-null    object
 11  Keywords                  828 non-null    object
 12  Style                     828 non-null    int64 
dtypes: int64(1), object(12)
memory usage: 90.6+ KB


In [101]:
print(df["Authors"].sample(1).tolist()[0])

Gährs, Swantje and Knoefel, Jan


In [107]:
for a in df["References"].sample(1).tolist()[0]:
    print(a)

 Aarakit, S.M., Ntayi, J.M., Wasswa, F., Buyinza, F., Adaramola, M.S., Ssennono, V.F.,  2022. The role of financial inclusion in adoption of solar photovoltaic systems: a case  of Uganda. Renew. Energy 198, 984998. https://doi.org/10.1016/j.  renene.2022.08.056. 
Best, R., 2022. Energy inequity variation across contexts. Appl. Energy 309, 118451.  https://doi.org/10.1016/j.apenergy.2021.118451. 
Best, R., Burke, P.J., Nishitateno, S., 2019. Understanding the determinants of rooftop  solar installation: evidence from household surveys in Australia. Aust. J. Agric.  Resour. Econ. 63, 922939. https://doi.org/10.1111/1467-8489.12319. 
Best, R., Chareunsy, A., Li, H., 2021. Equity and effectiveness of Australian small-scale  solar schemes. Ecol. Econ. 180, 106890 https://doi.org/10.1016/j.  ecolecon.2020.106890. 
Best, R., Chareunsy, A., 2022. The impact of income on household solar panel uptake:  exploring diverse results using Australian data. Energy Econ. 112, 106124 https://  doi.org/10

In [ ]:
"""
Skripta koja generira težinski graf iz hashtagova
te zapisuje najveću povezanu komponentu u 
"largest_cc.txt" datoteku
"""
import pandas as pd
from functions import stringlist_tolist, count_hash, remove_emoji, print_graph_info, pyvis_show
import networkx as nx
from itertools import combinations
import matplotlib.pyplot as plt
from pyvis.network import Network
import pickle
from tqdm import tqdm

N = 1000

print("Ucitavanje dataseta ...")
df = pd.read_csv("UR_6.csv", lineterminator="\n", nrows=N)
print("Dataset shape: ", df.shape)

print("Izdvajanje hashtagova ...")
df['hashtags'] = df['hashtags'].apply(stringlist_tolist)
indexer = df[df['hashtags'].apply(count_hash) >= 2]
hashes = indexer['hashtags']

print("Stvaranje grafa: ")
G = nx.Graph()
for line in tqdm(hashes):
    temp = list(combinations(set(line), 2))
    for comb in temp:
        comb = (remove_emoji(comb[0]), remove_emoji(comb[1]))
        if G.has_edge(comb[0], comb[1]):
            G[comb[0]][comb[1]]['weight'] += 1
        elif G.has_edge(comb[1], comb[0]):
            G[comb[1]][comb[0]]['weight'] += 1
        else:
            G.add_edge(comb[0], comb[1], weight=1)

largest_cc = max(nx.connected_components(G), key=len)
S = G.subgraph(largest_cc).copy()

print_graph_info(G)
print_graph_info(S)

# # Graf save
# pickle.dump(G, open('graph_pro14.pickle', 'wb'))
# pickle.dump(S, open('largest_cc_pro14.pickle', 'wb'))
# with open('largest_cc_pro14.txt', 'w') as f:
#     f.writelines(largest_cc)

print("Crtanje ...")
pyvis_show(S, True)